<a href="https://colab.research.google.com/github/NataliiaFakas/TFG_BMW/blob/main/7_GB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gradient Boosting


---


## 1. Cargar los datos

In [1]:
import pandas as pd
import numpy as np

from google.colab import drive
drive.mount('/content/drive')

ruta_macro = "/content/drive/MyDrive/TFG_BMW/datasets/bmw_dataset_variables_macro.csv"
ruta_bmw = "/content/drive/MyDrive/TFG_BMW/datasets/ventas_bmw.csv"

df_macro = pd.read_csv(ruta_macro)
df_bmw = pd.read_csv(ruta_bmw)

display(df_macro.head())
display(df_bmw.head())

Mounted at /content/drive


,Year,Unemployment_Rate,GDP_Growth,Deposit_Facility,HICP,Industrial_Production,Retail_Sales_Growth,Consumer_Confidence,Compensation_Per_Employee,Brent_Oil_Price,EUR_USD,Euribor_12M,Population_Total
0,2005,8.9,1.7,1.25,2.18,153.2,0.03,-4.2,31.22,54.57,1.2441,2.19,434585887
1,2006,8.2,3.2,2.50,2.19,147.7,0.04,-3.5,31.95,65.16,1.2556,3.08,436041018
2,2007,7.2,2.9,3.00,2.13,145.5,0.03,-2.1,32.75,72.44,1.3705,4.24,437514477
3,2008,7.1,0.4,2.00,3.30,148.0,0.01,-10.8,33.86,96.94,1.4708,4.63,439074280
4,2009,9.0,-4.4,0.25,0.29,144.5,-0.03,-21.7,34.41,61.74,1.3948,1.31,440467898


,Año,Ventas_BMW_Unidades
0,2005,1126768
1,2006,1185088
2,2007,1276793
3,2008,1202239
4,2009,1068770


## 2. Seleccionar las variables del modelo

In [2]:
variables_modelo = [
    "Year",
    "Compensation_Per_Employee",
    "Industrial_Production",
    "Population_Total",
    "EUR_USD"
]

df_macro_reducido = df_macro[variables_modelo]

display(df_macro_reducido.head())

,Year,Compensation_Per_Employee,Industrial_Production,Population_Total,EUR_USD
0,2005,31.22,153.2,434585887,1.2441
1,2006,31.95,147.7,436041018,1.2556
2,2007,32.75,145.5,437514477,1.3705
3,2008,33.86,148.0,439074280,1.4708
4,2009,34.41,144.5,440467898,1.3948


## 3. Unir las variables macroeconómicas con las ventas de BMW

In [4]:
def añadir_variables_macro(df_bmw, df_macro):

    df_final = df_bmw.merge(
        df_macro[variables_modelo],
        on="Year",
        how="left"
    )

    return df_final

df_bmw_filtrado = df_bmw.rename(
    columns={"Año": "Year"}
)

data = añadir_variables_macro(
    df_bmw_filtrado,
    df_macro_reducido
)

display(data.head())

print("Shape del dataset definitivo:", data.shape)

,Year,Ventas_BMW_Unidades,Compensation_Per_Employee,Industrial_Production,Population_Total,EUR_USD
0,2005,1126768,31.22,153.2,434585887,1.2441
1,2006,1185088,31.95,147.7,436041018,1.2556
2,2007,1276793,32.75,145.5,437514477,1.3705
3,2008,1202239,33.86,148.0,439074280,1.4708
4,2009,1068770,34.41,144.5,440467898,1.3948


Shape del dataset definitivo: (21, 6)


## 4. Comprobar valores nulos

In [5]:
print("Valores nulos por variable:")
display(data.isnull().sum())

Valores nulos por variable:


,0
Year,0
Ventas_BMW_Unidades,0
Compensation_Per_Employee,0
Industrial_Production,0
Population_Total,0
EUR_USD,0


## 5. Partición temporal

Utilizamos exactamente la misma división que en Random Forest y Ridge:

- Train: 2005–2018
- Dev: 2019–2021
- Test: 2022–2025

In [6]:
train = data[
    data["Year"] <= 2018
].copy()

dev = data[
    (data["Year"] >= 2019) &
    (data["Year"] <= 2021)
].copy()

test = data[
    data["Year"] >= 2022
].copy()

print("Observaciones Train:", len(train))
print("Observaciones Dev:", len(dev))
print("Observaciones Test:", len(test))

Observaciones Train: 14
Observaciones Dev: 3
Observaciones Test: 4


## 6. Crear las variables X e y

In [7]:
X_train = train.drop(
    columns=["Year", "Ventas_BMW_Unidades"]
)

y_train = train["Ventas_BMW_Unidades"]

X_dev = dev.drop(
    columns=["Year", "Ventas_BMW_Unidades"]
)

y_dev = dev["Ventas_BMW_Unidades"]

X_test = test.drop(
    columns=["Year", "Ventas_BMW_Unidades"]
)

y_test = test["Ventas_BMW_Unidades"]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_dev:", X_dev.shape)
print("y_dev:", y_dev.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (14, 4)
y_train: (14,)
X_dev: (3, 4)
y_dev: (3,)
X_test: (4, 4)
y_test: (4,)


## 7. Preparar los datos para TimeSeriesSplit

Para seleccionar los hiperparámetros utilizaremos todos los datos disponibles hasta 2021, manteniendo test separado.

In [8]:
RANDOM_STATE = 42

data_train_dev = data[
    data["Year"] <= 2021
].copy()

X_train_dev = data_train_dev.drop(
    columns=["Year", "Ventas_BMW_Unidades"]
)

y_train_dev = data_train_dev["Ventas_BMW_Unidades"]

## 8. Crear TimeSeriesSplit

In [9]:
from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(
    n_splits=4
)

for i, (train_index, val_index) in enumerate(
    tscv.split(X_train_dev),
    start=1
):

    print(
        f"Fold {i}: "
        f"Train = {len(train_index)} observaciones, "
        f"Validación = {len(val_index)} observaciones"
    )

Fold 1: Train = 5 observaciones, Validación = 3 observaciones
Fold 2: Train = 8 observaciones, Validación = 3 observaciones
Fold 3: Train = 11 observaciones, Validación = 3 observaciones
Fold 4: Train = 14 observaciones, Validación = 3 observaciones


## 9. Crear el modelo Gradient Boosting

In [10]:
from sklearn.ensemble import GradientBoostingRegressor

gb = GradientBoostingRegressor(
    random_state=RANDOM_STATE
)

## 10. Definir los hiperparámetros

Vamos a probar diferentes valores de:

- n_estimators: número de árboles.
- learning_rate: velocidad de aprendizaje.
- max_depth: profundidad máxima de los árboles.

In [11]:
param_grid_gb = {
    "n_estimators": [
        10,
        25,
        50,
        100
    ],

    "learning_rate": [
        0.01,
        0.05,
        0.1
    ],

    "max_depth": [
        1,
        2,
        3
    ]
}

## 11. Búsqueda de los mejores hiperparámetros

In [12]:
from sklearn.model_selection import GridSearchCV

grid_gb = GridSearchCV(
    estimator=gb,
    param_grid=param_grid_gb,
    cv=tscv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

grid_gb.fit(
    X_train_dev,
    y_train_dev
)

GridSearchCV(cv=TimeSeriesSplit(gap=0, max_train_size=None, n_splits=4, test_size=None),
             estimator=GradientBoostingRegressor(random_state=42), n_jobs=-1,
             param_grid={'learning_rate': [0.01, 0.05, 0.1],
                         'max_depth': [1, 2, 3],
                         'n_estimators': [10, 25, 50, 100]},
             scoring='neg_root_mean_squared_error')

## 12. Mostrar los mejores hiperparámetros

In [13]:
print("Mejores parámetros Gradient Boosting:")
print(grid_gb.best_params_)

print(
    "\nMejor RMSE medio de validación:"
)

print(
    -grid_gb.best_score_
)

Mejores parámetros Gradient Boosting:
{'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100}

Mejor RMSE medio de validación:
216614.27582416683


## 13. Obtener el modelo definitivo

In [14]:
modelo_gb = grid_gb.best_estimator_

modelo_gb.fit(
    X_train_dev,
    y_train_dev
)

GradientBoostingRegressor(random_state=42)

## 14. Realizar las predicciones sobre el conjunto de test

In [16]:
pred_test_gb = modelo_gb.predict(
    X_test
)

predicciones_gb = test[
    ["Year", "Ventas_BMW_Unidades"]
].copy()

predicciones_gb["Prediccion_GB"] = pred_test_gb

display(predicciones_gb)

,Year,Ventas_BMW_Unidades,Prediccion_GB
17,2022,2100692,2.170307e+06
18,2023,2253835,2.104102e+06
19,2024,2200177,2.104102e+06
20,2025,2169761,2.104102e+06


## 15. Calcular MSE

In [17]:
from sklearn.metrics import mean_squared_error

mse_test_gb = mean_squared_error(
    y_test,
    pred_test_gb
)

print(
    f"MSE Gradient Boosting en test: "
    f"{mse_test_gb:,.2f}"
)

MSE Gradient Boosting en test: 10,201,965,929.32


## 16. Calcular RMSE

In [18]:
rmse_test_gb = np.sqrt(
    mse_test_gb
)

print(
    f"RMSE Gradient Boosting en test: "
    f"{rmse_test_gb:,.2f}"
)

RMSE Gradient Boosting en test: 101,004.78


## 17. Calcular MAE

In [19]:
from sklearn.metrics import mean_absolute_error

mae_test_gb = mean_absolute_error(
    y_test,
    pred_test_gb
)

print(
    f"MAE Gradient Boosting en test: "
    f"{mae_test_gb:,.2f}"
)

MAE Gradient Boosting en test: 95,270.64


## 18. Calcular R²

In [20]:
from sklearn.metrics import r2_score

r2_test_gb = r2_score(
    y_test,
    pred_test_gb
)

print(
    f"R² Gradient Boosting en test: "
    f"{r2_test_gb:.4f}"
)

R² Gradient Boosting en test: -2.3317


## 19. Obtener las cuatro métricas juntas

In [21]:
resultado_gb = pd.DataFrame({
    "Modelo": [
        "Gradient Boosting"
    ],
    "MAE": [
        mae_test_gb
    ],
    "RMSE": [
        rmse_test_gb
    ],
    "R²": [
        r2_test_gb
    ]
})

display(resultado_gb)

,Modelo,MAE,RMSE,R²
0,Gradient Boosting,95270.642245,101004.781715,-2.331708
